<a href="https://colab.research.google.com/github/pragatiagarwal802-wq/surat-ev-infrastructure-analysis/blob/main/FINALTTT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# STEP 1: INSTALL LIBRARIES
# ==========================================
!pip install h3 pandas scikit-learn numpy

# ==========================================
# STEP 2: IMPORT LIBRARIES
# ==========================================
import pandas as pd
import numpy as np
import h3
import json
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import DBSCAN
from google.colab import files
import os

# ==========================================
# STEP 3: UPLOAD FILE (IMPORTANT FIX)
# ==========================================
print("📂 Please upload SURAT_EV_MASTER_COMPLETE.csv")
uploaded = files.upload()

# Get uploaded filename automatically
file_name = list(uploaded.keys())[0]
print(f"✅ Uploaded file: {file_name}")

# ==========================================
# ELITE PIPELINE CLASS
# ==========================================
class SuratEVElitePipeline:
    def __init__(self, file_path):
        self.df = pd.read_csv(file_path)
        print(f"🚀 Loaded {len(self.df)} hex cells.")

    def decay_function(self, x, alpha=1.5):
        return np.exp(-alpha * x)

    def process(self, weights,
                population_threshold=100000,
                hotspot_eps=0.01,
                hotspot_min_samples=8):

        df = self.df.copy()

        # ==============================
        # 1️⃣ NORMALIZE LAYERS
        # ==============================
        layers = [
            'population_5km_buffer',
            'traffic_density_score',
            'grid_score',
            'competitor_count',
            'flood_risk'
        ]

        for col in layers:
            scaler = MinMaxScaler()
            df[f'norm_{col}'] = scaler.fit_transform(df[[col]])

        # ==============================
        # 2️⃣ COMPETITION DECAY
        # ==============================
        df['competition_effect'] = self.decay_function(
            df['norm_competitor_count']
        )

        # ==============================
        # 3️⃣ FLOOD PENALTY
        # ==============================
        df['flood_effect'] = 1 - df['norm_flood_risk']

        # ==============================
        # 4️⃣ CONFIGURABLE SCORING
        # ==============================
        df['raw_score'] = (
            weights.get('pop', 0.4) * df['norm_population_5km_buffer'] +
            weights.get('traffic', 0.3) * df['norm_traffic_density_score'] +
            weights.get('grid', 0.2) * df['norm_grid_score'] +
            weights.get('comp', 0.05) * df['competition_effect'] +
            weights.get('flood', 0.05) * df['flood_effect']
        )

        df['readiness_score'] = df['raw_score'].clip(0, 1) * 100

        # ==============================
        # 5️⃣ HARD POPULATION THRESHOLD
        # ==============================
        df.loc[
            df['population_5km_buffer'] < population_threshold,
            'readiness_score'
        ] = 0

        # ==============================
        # 6️⃣ TIER CLASSIFICATION
        # ==============================
        def assign_tier(score):
            if score >= 75:
                return "Tier 1: Mega Hub"
            elif score >= 50:
                return "Tier 2: Standard Station"
            elif score > 0:
                return "Tier 3: Local Point"
            else:
                return "Not Suitable"

        df['site_tier'] = df['readiness_score'].apply(assign_tier)

        # ==============================
        # 7️⃣ HOTSPOT DETECTION (DBSCAN)
        # ==============================
        top_sites = df[df['readiness_score'] > 70].copy()

        top_sites['lat'] = top_sites['hex_id'].apply(
            lambda x: h3.cell_to_latlng(x)[0]
        )
        top_sites['lng'] = top_sites['hex_id'].apply(
            lambda x: h3.cell_to_latlng(x)[1]
        )

        clustering = DBSCAN(
            eps=hotspot_eps,
            min_samples=hotspot_min_samples
        ).fit(top_sites[['lat', 'lng']])

        top_sites['cluster'] = clustering.labels_

        df['hotspot_cluster'] = -1
        df.loc[top_sites.index, 'hotspot_cluster'] = top_sites['cluster']
        df['is_hotspot'] = df['hotspot_cluster'] >= 0

        # ==============================
        # 8️⃣ EXPLAINABLE BREAKDOWN
        # ==============================
        df['explanation'] = (
            "Pop:" + (df['norm_population_5km_buffer']*100).round(1).astype(str) +
            " | Traffic:" + (df['norm_traffic_density_score']*100).round(1).astype(str) +
            " | Grid:" + (df['norm_grid_score']*100).round(1).astype(str)
        )

        # ==============================
        # 9️⃣ GEOJSON EXPORT (TOP 2000)
        # ==============================
        features = []
        map_subset = df.nlargest(2000, 'readiness_score')

        for _, row in map_subset.iterrows():
            boundary = h3.cell_to_boundary(row['hex_id'])
            coords = [[p[1], p[0]] for p in boundary]
            coords.append(coords[0])

            features.append({
                "type": "Feature",
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [coords]
                },
                "properties": {
                    "hex_id": row['hex_id'],
                    "score": round(row['readiness_score'], 1),
                    "tier": row['site_tier'],
                    "hotspot": bool(row['is_hotspot']),
                    "cluster_id": int(row['hotspot_cluster']),
                    "explanation": row['explanation']
                }
            })

        geojson = {
            "type": "FeatureCollection",
            "features": features
        }

        # Save outputs
        df.to_csv("SURAT_EV_ELITE_DATA.csv", index=False)
        with open("SURAT_EV_ELITE_MAP.geojson", "w") as f:
            json.dump(geojson, f)

        print("\n✅ ELITE PIPELINE COMPLETE")
        print("Generated files:")
        print("1️⃣ SURAT_EV_ELITE_DATA.csv")
        print("2️⃣ SURAT_EV_ELITE_MAP.geojson")

        return df


# ==========================================
# STEP 4: RUN PIPELINE
# ==========================================
weights = {
    'pop': 0.40,
    'traffic': 0.25,
    'grid': 0.20,
    'comp': 0.10,
    'flood': 0.05
}

pipeline = SuratEVElitePipeline(file_name)
final_df = pipeline.process(weights)

# ==========================================
# STEP 5: DOWNLOAD OUTPUT FILES
# ==========================================
files.download("SURAT_EV_ELITE_DATA.csv")
files.download("SURAT_EV_ELITE_MAP.geojson")

📂 Please upload SURAT_EV_MASTER_COMPLETE.csv


Saving SURAT_EV_MASTER_COMPLETE (1).csv to SURAT_EV_MASTER_COMPLETE (1) (1).csv
✅ Uploaded file: SURAT_EV_MASTER_COMPLETE (1) (1).csv
🚀 Loaded 45881 hex cells.

✅ ELITE PIPELINE COMPLETE
Generated files:
1️⃣ SURAT_EV_ELITE_DATA.csv
2️⃣ SURAT_EV_ELITE_MAP.geojson


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>